# Step 1: SAMap inputs

One file per species with raw counts in `X` and the labels in obs. No `.raw`: SAMap would read it instead of `X`.

In [ ]:
OUT = ""  # run folder
HUMAN_H5AD = "/n/netscratch/eddy_lab/Everyone/nhilgert/final_objects/Cevrim_Human_Only_Fib_Dec_Cells.h5ad"
MOUSE_H5AD = "/n/netscratch/eddy_lab/Everyone/nhilgert/final_objects/Cevrim_XMens_Only_Fib_Dec_Cells.h5ad"
COUNTS_LAYER = "counts"  # layer with the raw counts

In [ ]:
import datetime
import hashlib
import json
import os
from pathlib import Path

import anndata as ad
import numpy as np
import scipy.sparse as sp

In [ ]:
# SAMap's species codes.
SOURCES = {"hs": HUMAN_H5AD, "mm": MOUSE_H5AD}
FIBDEC = {"decidual_cell", "endometrial_fibroblast"}
# The objects label the compartments eFib and dec; later steps use the long names.
LABEL_MAP = {"eFib": "endometrial_fibroblast", "dec": "decidual_cell"}

# SAMap's evaluation in step 2 reads these two column names.
OBS_COLUMNS = ["sample", "timepoint", "cell_type", "cell_subtype"]

assert OUT, "set OUT, the run folder"
out_dir = Path(OUT) / "1_inputs"
partial_dir = Path(OUT) / "1_inputs.partial"
assert not out_dir.exists(), f"{out_dir} exists; step 1 is done"
partial_dir.mkdir(parents=True, exist_ok=True)


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 24), b""):
            digest.update(block)
    return digest.hexdigest()


def samap_input(source):
    """One species' SAMap input: raw counts in X (float32), the labels, and var as stored."""
    final = ad.read_h5ad(source)
    final.obs["cell_type"] = final.obs["cell_type"].astype(str).map(lambda v: LABEL_MAP.get(v, v)).astype("category")
    not_fibdec = ~final.obs["cell_type"].astype(str).isin(FIBDEC)
    assert not not_fibdec.any(), f"{int(not_fibdec.sum())} nuclei of {source} are not fibroblast or decidual"

    obs = final.obs[OBS_COLUMNS].copy()
    obs["cell_type_annotations"] = obs["cell_type"].astype(str)
    obs["cell_subtype_annotations"] = obs["cell_subtype"].astype(str)
    counts = sp.csr_matrix(final.layers[COUNTS_LAYER], dtype=np.float32)
    return ad.AnnData(X=counts, obs=obs, var=final.var.copy())


def count_summary(adata):
    """Check that X holds raw counts: integers, count-sized values, library sizes that vary between nuclei."""
    library = np.asarray(adata.X.sum(axis=1)).ravel()
    summary = {
        "nuclei": adata.n_obs,
        "genes": adata.n_vars,
        "integer": bool(np.array_equal(adata.X.data, np.round(adata.X.data))),
        "max": float(adata.X.max()),
        "library_median": float(np.median(library)),
        "library_cv": round(float(library.std() / library.mean()), 3),
        "nuclei_without_counts": int((library == 0).sum()),
    }
    is_counts = summary["integer"] and summary["max"] > 50 and summary["library_cv"] > 0.05
    assert is_counts and summary["nuclei_without_counts"] == 0, f"X does not look like raw counts: {summary}"
    return summary

In [ ]:
record = {"written": datetime.datetime.now().isoformat(timespec="seconds")}
for species, source in SOURCES.items():
    adata = samap_input(source)
    summary = count_summary(adata)
    adata.uns["samap_input"] = f"raw counts of {source}"
    adata.write_h5ad(partial_dir / f"{species}_fib_dec_all_raw.h5ad")
    record[species] = {"source": str(source), "source_sha256": sha256_of(source), **summary}
    print(species, json.dumps(record[species]))

(partial_dir / "inputs.json").write_text(json.dumps(record, indent=1) + "\n")
os.replace(partial_dir, out_dir)
print(f"wrote {out_dir}: {sorted(p.name for p in out_dir.iterdir())}")